In [ ]:
# Crawl outputs of open-gira TC/grid analysis, specifically:
# power/by_storm_set/{STORM_SET}/disruption/pop_affected_RP/{RETURN_PERIOD}.gpq files
# for each return period, draw a 4 panel map of affected regions with 1 panel per STORM/IRIS present/future

In [ ]:
from collections import defaultdict
from glob import glob
from pathlib import Path
import os
import warnings
warnings.simplefilter("ignore", UserWarning)

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D
import matplotlib.ticker
import matplotlib.colors
import numpy as np

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
results_dir = "/data/tc-grid/open-gira/results"
targets_path = "power/targets.geoparquet"  # within results_dir
country_polygons_path = "input/admin-boundaries/admin-level-0.geoparquet"  # within results_dir

# Output paths
plot_dir = Path("../plots")

# Mapping from region display name to region bounding box
regions = {
    "Americas": [-113, -3, -57, 53],
    "Indian Ocean": [36, -29, 99, 34],
    "West Pacific": [100, -53, 155, 57],
}
# Mapping from open-gira STORM_SET to display name (future STORM will be aggregated)
atmospheres_to_scenario = {
    "STORM_constant": "STORM 2020",
    "STORM_CNRM-CM6-1-HR": "STORM 2050 RCP8.5",
    "STORM_CMCC-CM2-VHR4": "STORM 2050 RCP8.5",
    "STORM_EC-Earth3P-HR": "STORM 2050 RCP8.5",
    "STORM_HadGEM3-GC31-HM": "STORM 2050 RCP8.5",
    "IRIS_PRESENT": "IRIS 2020",
    "IRIS_SSP5-2050": "IRIS 2050 SSP5-8.5"
}
# Scenarios to plot
scenarios = list(sorted(set(atmospheres_to_scenario.values())))

# Region aspect ratios must be appropriate for plot layout, i.e. square for a square plot
# Absolute sizes of bbox do not matter
print("Region,\t\txspan, yspan, ratio")
for region, bbox in regions.items():
    xmin, ymin, xmax, ymax = bbox
    xspan = xmax - xmin
    yspan = ymax - ymin
    print(f"{region},\t{xspan:.2f}, {yspan:.2f}, {xspan/yspan:.2f}")

In [ ]:
# Estimate population connected to grid in each country
targets = gpd.read_parquet(os.path.join(results_dir, targets_path), columns=["population", "iso_a3", "geometry"])
targets["geometry"] = targets.geometry.representative_point()

# Assemble optimum country to threshold mapping
# Calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# Thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)
# Global default threshold, ms-1
default_threshold = 30.0

atmosphere_to_disruption = []
for atmosphere in atmospheres_to_scenario.keys():
    path = os.path.join(results_dir, f"power/by_storm_set/{atmosphere}/disruption/EAPA_admin-level-2-0.gpq")
    df = gpd.read_parquet(path)
    
    # Set countries with no calibrated value to the global default
    df["pop_at_risk"] = df[f"{default_threshold:.1f}"]
    
    valid_countries = thresholds.index.unique()
    df = df[df.ISO_A3.isin(valid_countries)]
    df = df.merge(
        thresholds[['threshold_ms-1']],
        left_on='ISO_A3',
        right_index=True,
        how='left'
    )
    
    df['pop_at_risk'] = df.apply(
        lambda row: row[str(row['threshold_ms-1'])] if pd.notna(row['threshold_ms-1']) else np.nan,
        axis=1
    )
        
    # Drop the now redundant master set of thresholds
    cols_to_keep = ['GID_0', 'GID_1', 'GID_2', 'ISO_A3', 'NAME_0', 'NAME_1', 'NAME_2', 'geometry', 'pop_at_risk']
    
    # Ideally, I wouldn't fillna here, but if I remove this (and handle the zero division error later)
    # the results are not what I'd expect... so I'm leaving it and not investigating any further for now
    df["pop_at_risk"] = df["pop_at_risk"].fillna(0)
    
    df = df.loc[:, cols_to_keep].reset_index(drop=True)
    
    df = df.sort_values("pop_at_risk", ascending=False)
    
    df["atmosphere"] = atmosphere
    df["GID"] = df.apply(lambda row: f"{row.GID_0}_{row.GID_1}_{row.GID_2}", axis=1)
    atmosphere_to_disruption.append(df)
    
all_atmospheres = pd.concat(atmosphere_to_disruption)
all_atmospheres["scenario"] = all_atmospheres.atmosphere.map(atmospheres_to_scenario)

storm_future_mask = all_atmospheres.scenario == "STORM 2050 RCP8.5"
storm_future_pop_at_risk = all_atmospheres.loc[storm_future_mask, ["GID", "pop_at_risk"]].groupby("GID").mean()
storm_future_mean = all_atmospheres[storm_future_mask].drop_duplicates("GID") \
    .drop(columns=["pop_at_risk", "atmosphere"]).set_index("GID").join(storm_future_pop_at_risk)
concat = pd.concat([all_atmospheres[~storm_future_mask], storm_future_mean])
concat["GID"] = concat.apply(lambda row: f"{row.GID_0}_{row.GID_1}_{row.GID_2}", axis=1)
df = concat.sort_values(["GID", "scenario"]).set_index("GID") 

# Duplicate regions, each with different target
region_to_target = df.sjoin(targets, how="left")
# Group by region, sum target populations to find connected population per region
population = region_to_target[["population"]].groupby(region_to_target.index).sum()
df = df.join(population)

# Calculate a measure normalised by the regional (connected) population
df["pop_at_risk_pc"] = df.pop_at_risk / df.population

# We can't log plot a zero value, so nudge 0's into positive
nudge = 1E-9
df.loc[df.pop_at_risk_pc == 0, "pop_at_risk_pc"] = nudge
df.loc[df.pop_at_risk == 0, "pop_at_risk"] = nudge

# Determine whether network representations pass muster
node_paths = glob(os.path.join(results_dir, "power/by_country/*/network/nodes.geoparquet"))
iso_a3_grid_state = {}
for path in node_paths:
    iso_a3 = path.split("/")[-3]
    print(iso_a3, end="\r")
    
    try:
        nodes = pd.read_parquet(path, columns=["asset_type", "power_mw"])
    except ValueError:
        # empty node file
        iso_a3_grid_state[iso_a3] = False
        continue

    source_nodes = nodes[nodes.asset_type == "source"]
    target_nodes = nodes[nodes.asset_type == "target"]
    operating = ((source_nodes.power_mw.sum() > 0) and (target_nodes.power_mw.sum() < 0))
    iso_a3_grid_state[iso_a3] = operating
    
grids = pd.Series(index=iso_a3_grid_state.keys(), data=iso_a3_grid_state.values(), name="operating")
bad_grids = grids[~grids].index

# Set results for bad grids to NaN
df.loc[df.ISO_A3.isin(bad_grids), ["pop_at_risk", "pop_at_risk_pc"]] = np.nan

# If any countries are missing from our dataset, add them so we can explicitly mark as missing
countries = gpd.read_parquet(os.path.join(results_dir, country_polygons_path))
countries.geometry = countries.geometry.boundary
missing_countries = list(set(countries.GID_0) - set(df.ISO_A3))
# The missing countries will have NaN valued pop_at_risk and pop_at_risk_pc 
df = pd.concat([df, countries.set_index("GID_0").loc[missing_countries]])

df = df.set_index([df.index, df.scenario])

In [ ]:
# Draw one scenario (3 map views) per plot, save to disk
# We will later arrange these prerendered plots into a larger panel
# You know.. Ghent alterpiece by van Eyck...

def shift_lon(geom):
    def shift(x, y, z=None):
        x = np.where(x < 0, x + 360, x)
        return x, y
    return transform(shift, geom)

def safe_path(path: str) -> str:
    return "_".join(path.strip().split()).lower()

def draw_map(
    df: gpd.GeoDataFrame,
    countries: gpd.GeoDataFrame,
    scenario: str,
    region: str,
    regions: dict[str, list[float, float, float, float]],
    ax: plt.axis,
    norm,
    scalar_mappable,
    cmap,
) -> plt.axis:

    xmin, ymin, xmax, ymax = regions[region]
    disruption = df.xs(scenario, level=1).cx[xmin: xmax, ymin: ymax]
    country_boundaries = countries.cx[xmin: xmax, ymin: ymax]
    missing_kwds = {"hatch": "///", "edgecolor": "black", "color": "white", "facecolor": "white", "linewidth": 0.2, "alpha": 0.4}
    disruption.plot("pop_at_risk_pc", ax=ax, legend=False, cmap=cmap, norm=norm, missing_kwds=missing_kwds)
    country_boundaries.plot(ax=ax, linewidth=0.5, color="grey", alpha=0.5)
    ax.grid(alpha=0.2)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.xaxis.set_major_locator(matplotlib.ticker.MultipleLocator(base=10))
    ax.yaxis.set_major_locator(matplotlib.ticker.MultipleLocator(base=10))
    ax.tick_params(bottom=False, top=False, left=False, right=False)
    ax.tick_params(labelbottom=False, labeltop=False, labelleft=False, labelright=False)
    ax.set_aspect(1)  # essential to keep plots neatly tiled
    return ax

for scenario in scenarios:
    matplotlib.rcParams['hatch.linewidth'] = 0.5
    f, axes = plt.subplot_mosaic(
        [
            ["Americas", "West Pacific"],
            ["Indian Ocean", "West Pacific"]
        ],
        figsize=(6, 6),
        layout="compressed",
    )
    cmap = plt.get_cmap("YlOrBr")
    cmap.set_under("white")
    norm = matplotlib.colors.LogNorm(vmin=3E-3, vmax=df.pop_at_risk_pc.max())
    scalar_mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    
    for region, ax in axes.items():
        draw_map(df, countries, scenario, region, regions, ax, norm, scalar_mappable, cmap)
    
    path = plot_dir / f"{safe_path(scenario)}.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close(f)

In [ ]:
f, axes = plt.subplots(2, 2, figsize=(9, 8), dpi=254)

# This axis is used to create another, cax, where we render the colourbar
encompassing_ax = f.add_axes([0.03, 0.06, 0.87, 0.88])
encompassing_ax.axis("off")
divider = make_axes_locatable(encompassing_ax)
cax = divider.append_axes("right", size="2%")
cbar = f.colorbar(
    scalar_mappable,
    cax=cax,
    extend="min",
    label="Proportion of power-connected population affected"
)
cax.yaxis.set_major_formatter(FormatStrFormatter("%G"))
cax.yaxis.set_minor_formatter(FormatStrFormatter("%G"))
cax.tick_params(axis='y', which='minor', labelsize=8)

plt.subplots_adjust(left=0.05, right=0.9, bottom=0.05, top=0.95, wspace=0, hspace=0)
for i, scenario in enumerate(scenarios):
    img = plt.imread(plot_dir / f"{safe_path(scenario)}.png")
    ax = axes.ravel()[i]
    ax.imshow(img)
    ax.axis("off")

plt.subplots_adjust(hspace=0.05, right=0.88)

labelsize=10
f.text(0.26, 0.03, "Present day", transform=f.transFigure, ha="center", va="center", fontsize=labelsize)
f.text(0.67, 0.03, "2050 RCP8.5", transform=f.transFigure, ha="center", va="center", fontsize=labelsize)
f.text(0.04, 0.27, "STORM", transform=f.transFigure, ha="center", va="center", rotation=90, fontsize=labelsize)
f.text(0.04, 0.73, "IRIS", transform=f.transFigure, ha="center", va="center", rotation=90, fontsize=labelsize)

f.savefig(plot_dir / "EAPA_pcp_map_panel.png")
plt.close(f)